In [19]:
from pathlib import Path

from datasets import DatasetDict

from src.config.paths import DEFAULT_TRAIN_LLM_CONFIG_PATH
from src.ml.config.loader import load_experiment_config, load_llm_training_config
from src.ml.datasets.splits import Splits, build_splits
from src.ml.llm.dataset import (
    SPLIT_NAMES,
    build_llm_dataset,
    build_split_dataset,
    push_to_hub,
    summarize,
)

In [2]:
LLM_CONFIG = "llama31_8b_qlora"
args_config = f"configs/train_llm/{LLM_CONFIG}.yaml"

In [3]:
DEFAULT_TRAIN_LLM_CONFIG_PATH

PosixPath('/Users/felipeformenti/dev/fformenti/nba_bets/configs/train_llm/llama31_8b_qlora.yaml')

In [8]:
llm_config = load_llm_training_config(args_config)

In [12]:
llm_config

LLMTrainingConfig(description='QLoRA SFT of Llama-3.1-8B on nba-bets point-differential prompts', run_name=None, base_model='meta-llama/Meta-Llama-3.1-8B', hub=HubConfig(user='fformenti', project_name='nba-bets', private=True, push_checkpoints=True), data=LLMDataConfig(dataset_name='fformenti/nba-bets', train_split='train', eval_split='validation', prompt_field='prompt', max_sequence_length=1024, max_train_samples=None, source_experiment_config='configs/train/llm_features.yaml', serialization_format='labeled'), quantization=QuantizationConfig(mode='8bit', compute_dtype='bfloat16', bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True), lora=LoraSettings(r=32, alpha=64, dropout=0.1, target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'], bias='none'), training=LLMTrainerConfig(epochs=2, per_device_train_batch_size=2, gradient_accumulation_steps=4, learning_rate=0.0001, lr_scheduler_type='cosine', warmup_ratio=0.03, optimizer='paged_adamw_32bit', weight_decay=0.001, max_grad_norm=0.

In [10]:
experiment_config = None
if experiment_config is None:
    experiment_config = load_experiment_config(llm_config.data.source_experiment_config)


In [ ]:
experiment_config

ExperimentConfig(description='LLM feature set: all games, plus win/loss streak and raw home/road days', tags={'variation': 'llm_features'}, data=DataConfig(path='data/processed/regular_season/games_features.csv', target_column='win_bool', date_column='gameDate', drop_na=True), filters=FiltersConfig(min_season='1996/97', minimum_games_train=15, minimum_games_test=15), weighting=WeightingConfig(enabled=True, saturation_K=30), splitting=SplittingConfig(method='temporal', test_size=0.2, val_size=0.2, random_state=42, test_start_season='2024/25', val_seasons=3), feature_engineering=FeatureEngineeringConfig(selection_mode='inclusion', sos_adj_alpha=1.0, gds_beta=0.1, features=FeaturesMapConfig(record=FeatureGroupConfig(lags=[1, 3, 5, 8, 13, 21, 34, 55, 82], location_lags=[5, 10, 20, 41], delta=True, enabled=False), point_differential=FeatureGroupConfig(lags=[1, 3, 5, 8, 13, 21, 34, 55, 82], location_lags=[5, 10, 20, 41], delta=True, enabled=False), norm_point_differential=FeatureGroupConfig(

In [ ]:
splits = build_splits(experiment_config)

In [17]:
llm_config.data.serialization_format

'labeled'

In [20]:
dataset = DatasetDict(
    {
        name: build_split_dataset(splits, name, llm_config.data.serialization_format)
        for name in SPLIT_NAMES
    }
)

In [43]:
# push_to_hub(dataset, llm_config.data.dataset_name, private=llm_config.hub.private)


In [ ]:
dataset_train = dataset["train"]


In [38]:
dataset_train

Dataset({
    features: ['game_id', 'prompt', 'completion'],
    num_rows: 23247
})

In [41]:
print(dataset_train.data["prompt"][203])

You are responsible for predicting the outcome of a basketball game.
Below are the pre-game statistics for the home and visiting teams. Positive values favour the home team.

Game: Spurs (home) vs Clippers (away) — 1997-01-05, 1996/97

Recent form:
  normalized point differential, last 1 game: -0.01
  normalized point differential, last 3 games: 0
  normalized point differential, last 5 games: -0.02
  normalized point differential, last 8 games: 0.02
  normalized point differential, last 13 games: -0.01
  normalized point differential, last 21 games: 0
  normalized point differential, last 34 games: -0.01
  normalized point differential, last 55 games: -0.01
  normalized point differential, last 82 games: -0.01
  current win/loss streak: 1

Venue form (home team at home vs away team on the road):
  normalized point differential, last 5 games: 0.04
  normalized point differential, last 10 games: 0.03
  normalized point differential, last 20 games: 0
  normalized point differential, last

In [37]:
print(dataset_train.data["completion"][1])


+4


In [24]:
dataset_test = dataset["test"]

In [34]:
print(dataset_test.data["prompt"][1])

You are responsible for predicting the outcome of a basketball game.
Below are the pre-game statistics for the home and visiting teams. Positive values favour the home team.

Game: Magic (home) vs Pistons (away) — 2024-11-23, 2024/25

Recent form:
  normalized point differential, last 1 game: 0.01
  normalized point differential, last 3 games: -0.01
  normalized point differential, last 5 games: 0.01
  normalized point differential, last 8 games: 0.05
  normalized point differential, last 13 games: 0.01
  normalized point differential, last 21 games: 0.02
  normalized point differential, last 34 games: 0.02
  normalized point differential, last 55 games: 0.02
  normalized point differential, last 82 games: 0.02
  current win/loss streak: 3

Venue form (home team at home vs away team on the road):
  normalized point differential, last 5 games: 0.07
  normalized point differential, last 10 games: 0.06
  normalized point differential, last 20 games: 0.06
  normalized point differential, l